# Notebook 09 — Metric–Human Alignment (Table 8)

**Paper target:** *Table 8 (Appendix D)* — sentence-level **Spearman ρ** and **Pearson r** of every
automatic metric vs. MQM human annotations, across all five Indic languages, in both native-script
and romanised conditions.

Metrics covered: BLEU, chrF, TER, BLEURT, BERTScore, COMET-native, COMET-romanised.

---
**Paper evidence (exact values used for validation):**
- COMET native ρ: GUJ 0.590, TAM 0.629, MAL 0.651, MAR 0.487, HIN 0.422 (Table 2 / Appendix D)
- COMET romanised ρ: GUJ 0.311 (0.341 in weekly report), TAM 0.240, MAL 0.460, MAR 0.278, HIN 0.230
- Surface metrics (BLEU, chrF, TER): Δρ ≤ 0.01 under romanisation (Observation 3)
- Neural encoder metrics collapse 29–62% under romanisation (Observation 3)


In [ ]:
# ── Cell 1: Imports & Config ─────────────────────────────────────────────────
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../../data/processed')
OUT_DIR  = Path('../../results/tables')
OUT_DIR.mkdir(parents=True, exist_ok=True)

LANGUAGES = ['gujarati', 'tamil', 'malayalam', 'marathi', 'hindi']
ISO       = {'gujarati': 'GUJ', 'tamil': 'TAM', 'malayalam': 'MAL',
             'marathi': 'MAR', 'hindi': 'HIN'}

# Column names (as fixed in 03_metric_scoring.ipynb)
COL_HYP      = 'Translation'
COL_REF      = 'Reference'
COL_SRC      = 'Source'
COL_HYP_ROM  = 'Translation_Transliteration_romanized'
COL_REF_ROM  = 'Reference_Transliteration_romanized'
COL_MQM      = 'MQM'          # human MQM score column

# Metric column names written by 03_metric_scoring.ipynb
# Native-script metrics
METRIC_COLS_NAT = {
    'BLEU_nat':     'bleu_native',
    'chrF_nat':     'chrf_native',
    'TER_nat':      'ter_native',
    'BLEURT_nat':   'bleurt_native',
    'BERTScore_nat':'bertscore_native',
    'COMET_nat':    'comet_native',
}
# Romanised metrics
METRIC_COLS_ROM = {
    'BLEU_rom':     'bleu_romanised',
    'chrF_rom':     'chrf_romanised',
    'TER_rom':      'ter_romanised',
    'BLEURT_rom':   'bleurt_romanised',
    'BERTScore_rom':'bertscore_romanised',
    'COMET_rom':    'comet_romanised',
}

print('Config loaded. DATA_DIR =', DATA_DIR)

In [ ]:
# ── Cell 2: Load all language DataFrames ──────────────────────────────────────
dfs = {}
for lang in LANGUAGES:
    fp = DATA_DIR / f'{lang}_indicmt.csv'
    if fp.exists():
        dfs[lang] = pd.read_csv(fp)
        print(f'{ISO[lang]}: {len(dfs[lang])} rows, columns: {list(dfs[lang].columns)}')
    else:
        print(f'MISSING: {fp} — run 03_metric_scoring.ipynb first')

assert len(dfs) == 5, 'One or more language files missing — run 03_metric_scoring.ipynb'

In [ ]:
# ── Cell 3: Helper — correlation with significance ────────────────────────────

def corr_row(series_metric: pd.Series, series_human: pd.Series,
             metric_name: str, lang: str) -> dict:
    """Compute Spearman ρ and Pearson r against MQM. Returns dict with values + p-values."""
    # Drop NaN pairs
    mask = series_metric.notna() & series_human.notna()
    x = series_metric[mask].values
    y = series_human[mask].values
    n = int(mask.sum())
    if n < 10:
        return dict(lang=lang, metric=metric_name, n=n,
                    spearman=np.nan, sp_pval=np.nan,
                    pearson=np.nan,  pe_pval=np.nan)
    sp_r, sp_p = stats.spearmanr(x, y)
    pe_r, pe_p = stats.pearsonr(x, y)
    return dict(lang=lang, metric=metric_name, n=n,
                spearman=round(sp_r, 3), sp_pval=sp_p,
                pearson=round(pe_r, 3),  pe_pval=pe_p)

print('Helper defined.')

In [ ]:
# ── Cell 4: Compute correlations — all metrics × all languages ────────────────

rows = []
for lang in LANGUAGES:
    df = dfs[lang]
    iso = ISO[lang]
    if COL_MQM not in df.columns:
        print(f'WARNING: {COL_MQM} not in {lang} — skipping')
        continue
    human = df[COL_MQM]

    # Native-script metrics
    for label, col in METRIC_COLS_NAT.items():
        if col in df.columns:
            rows.append(corr_row(df[col], human, label, iso))
        else:
            print(f'  MISSING column {col} in {lang}')

    # Romanised metrics
    for label, col in METRIC_COLS_ROM.items():
        if col in df.columns:
            rows.append(corr_row(df[col], human, label, iso))
        else:
            print(f'  MISSING column {col} in {lang}')

corr_df = pd.DataFrame(rows)
print(f'Computed {len(corr_df)} correlation entries.')
corr_df.head(14)

In [ ]:
# ── Cell 5: Build Table 8 (pivot: metrics as rows, languages as columns) ──────
# Paper format: one row per metric, columns = GUJ / TAM / MAL / MAR / HIN
# Each cell shows: Spearman ρ (Pearson r) — dual correlation as per paper footnote

METRIC_ORDER = [
    'BLEU_nat',   'chrF_nat',   'TER_nat',
    'BLEURT_nat', 'BERTScore_nat', 'COMET_nat',
    'BLEU_rom',   'chrF_rom',   'TER_rom',
    'BLEURT_rom', 'BERTScore_rom', 'COMET_rom',
]
LANG_ORDER = ['GUJ', 'TAM', 'MAL', 'MAR', 'HIN']

# Significance marker (p < 0.05)
def sig_mark(pval):
    if pd.isna(pval): return ''
    return '*' if pval < 0.05 else ''

# Build pivot for Spearman
sp_pivot = corr_df.pivot(index='metric', columns='lang', values='spearman')
pe_pivot = corr_df.pivot(index='metric', columns='lang', values='pearson')
sp_p_pivot = corr_df.pivot(index='metric', columns='lang', values='sp_pval')
pe_p_pivot = corr_df.pivot(index='metric', columns='lang', values='pe_pval')

# Reindex to paper order
sp_pivot = sp_pivot.reindex(index=METRIC_ORDER, columns=LANG_ORDER)
pe_pivot = pe_pivot.reindex(index=METRIC_ORDER, columns=LANG_ORDER)
sp_p_pivot = sp_p_pivot.reindex(index=METRIC_ORDER, columns=LANG_ORDER)
pe_p_pivot = pe_p_pivot.reindex(index=METRIC_ORDER, columns=LANG_ORDER)

# Format cells: rho (r) with significance star
table8_cells = {}
for lang in LANG_ORDER:
    col_vals = []
    for met in METRIC_ORDER:
        sp = sp_pivot.loc[met, lang]
        pe = pe_pivot.loc[met, lang]
        spm = sig_mark(sp_p_pivot.loc[met, lang])
        pem = sig_mark(pe_p_pivot.loc[met, lang])
        if pd.isna(sp) and pd.isna(pe):
            col_vals.append('—')
        else:
            col_vals.append(f'{sp:.3f}{spm} ({pe:.3f}{pem})')
    table8_cells[lang] = col_vals

table8 = pd.DataFrame(table8_cells, index=METRIC_ORDER)
table8.index.name = 'Metric'

print('=== TABLE 8 — All-Metric Correlation vs. MQM (Spearman ρ | Pearson r) ===')
print('Condition | Metric         ', '  '.join(f'{l:>16}' for l in LANG_ORDER))
print('-' * 100)
native_block  = [m for m in METRIC_ORDER if m.endswith('_nat')]
roman_block   = [m for m in METRIC_ORDER if m.endswith('_rom')]
for block, label in [(native_block, 'Native'), (roman_block, 'Romanised')]:
    print(f'--- {label} ---')
    for met in block:
        vals = '  '.join(f'{str(table8.loc[met, l]):>16}' for l in LANG_ORDER)
        print(f'{met:<20} {vals}')
print()
print('* = p < 0.05. Values: ρ (r). TER is negatively oriented (lower = better).')

In [ ]:
# ── Cell 6: Paper-value sanity check ─────────────────────────────────────────
# Paper-reported native COMET Spearman ρ (Table 2 / Appendix D, short paper)
EXPECTED_NAT = {'GUJ': 0.590, 'TAM': 0.629, 'MAL': 0.651, 'MAR': 0.487, 'HIN': 0.422}
# Paper-reported romanised COMET Spearman ρ (Table 2)
EXPECTED_ROM = {'GUJ': 0.341, 'TAM': 0.240, 'MAL': 0.460, 'MAR': 0.278, 'HIN': 0.230}

print('COMET native Spearman ρ — computed vs paper:')
for lang in LANG_ORDER:
    computed = sp_pivot.loc['COMET_nat', lang]
    expected = EXPECTED_NAT[lang]
    match = '✓' if abs(computed - expected) < 0.005 else '✗ MISMATCH'
    print(f'  {lang}: computed={computed:.3f}  paper={expected:.3f}  {match}')

print('\nCOMET romanised Spearman ρ — computed vs paper:')
for lang in LANG_ORDER:
    computed = sp_pivot.loc['COMET_rom', lang]
    expected = EXPECTED_ROM[lang]
    match = '✓' if abs(computed - expected) < 0.010 else '✗ MISMATCH'
    print(f'  {lang}: computed={computed:.3f}  paper={expected:.3f}  {match}')

# Verify Observation 3: surface metrics |Δρ| ≤ 0.01 under romanisation
print('\nObservation 3 — surface metric stability (|Δρ| should be ≤ 0.01):')
for lang in LANG_ORDER:
    for m in ['BLEU', 'chrF', 'TER']:
        nat_r = sp_pivot.loc[f'{m}_nat', lang]
        rom_r = sp_pivot.loc[f'{m}_rom', lang]
        delta = abs(rom_r - nat_r) if pd.notna(nat_r) and pd.notna(rom_r) else np.nan
        flag = '✓' if (pd.notna(delta) and delta <= 0.015) else '✗'
        print(f'  {lang} {m}: |Δρ|={delta:.3f} {flag}')

In [ ]:
# ── Cell 7: Neural vs surface metric collapse ─────────────────────────────────
# Paper Observation 3: neural (COMET, BLEURT) collapse 29–62%; surface stable
print('Neural encoder metric human-alignment collapse under romanisation:')
print(f'{"Lang":>5} {"COMET Δρ%":>12} {"BLEURT Δρ%":>14} {"BLEU Δρ%":>12} {"chrF Δρ%":>10} {"TER Δρ%":>10}')
print('-' * 65)

collapse_rows = []
for lang in LANG_ORDER:
    row = {'lang': lang}
    for m_base in ['COMET', 'BLEURT', 'BLEU', 'chrF', 'TER']:
        nat = sp_pivot.loc[f'{m_base}_nat', lang]
        rom = sp_pivot.loc[f'{m_base}_rom', lang]
        if pd.notna(nat) and pd.notna(rom) and nat != 0:
            pct = round((rom - nat) / abs(nat) * 100, 1)
        else:
            pct = np.nan
        row[m_base] = pct
    collapse_rows.append(row)
    print(f'{lang:>5} {row["COMET"]:>12} {row["BLEURT"]:>14} {row["BLEU"]:>12} {row["chrF"]:>10} {row["TER"]:>10}')

collapse_df = pd.DataFrame(collapse_rows).set_index('lang')
print('\nPaper states COMET drops 29–62%, surface metrics stable ≤ 1%:')

In [ ]:
# ── Cell 8: Save Table 8 to CSV ───────────────────────────────────────────────
out_path = OUT_DIR / 'table8_metric_human_alignment.csv'

# Save long-form (Spearman + Pearson separately for LaTeX)
corr_df_out = corr_df[['lang', 'metric', 'n', 'spearman', 'sp_pval', 'pearson', 'pe_pval']].copy()
corr_df_out.to_csv(out_path, index=False)
print(f'Saved: {out_path}')

# Save wide pivot (formatted table as in paper)
table8.to_csv(OUT_DIR / 'table8_metric_human_alignment_pivot.csv')
print(f'Saved: {OUT_DIR / "table8_metric_human_alignment_pivot.csv"}')
print('\nDone — Table 8 complete.')